# Step 4c: INT4 sensitivity check on Qwen3-4B (RULER multikey, 16K)

**Why.** In the v2 RULER runs, Qwen3-4B was 200/200 for FP16, FP8 e4m3 and FP8 e5m2 on both tasks. That alone cannot tell "FP8 has no effect" from "the test cannot see a loss". The plan's step 4 says: if nothing degrades, add a lower-precision variant. INT4 is that variant: if it fails on the same prompts, the test can detect a cache loss on this model, and the FP8 result means FP8's loss is below what the test resolves.

**What runs.** Same notebook code and prompts as `step3-4_needle_v2_ruler/ruler_niah_qwen3-4b.ipynb` (same builder and seeds), only:
- task `multikey`, length 16K, 5 depths x 5 variants = 25 prompts;
- modes `fp16` and `int4` (per-token min-max INT4 on K and V, simulated like FP8: rounded at cache write, stored in FP16, prefill exact).

The FP16 file from the v2 run is reused: it is on the same Drive path, so the FP16 pass skips every prompt it already has and costs nothing. Only INT4 is new (about 25 trials of roughly 30 s).

In [ ]:
%pip install -q -U transformers accelerate datasets matplotlib pandas tqdm wonderwords

In [ ]:
import json, os, random, re, time, math
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Switch the runtime to a GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print(torch.__version__, GPU_NAME, f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

# Results live on Google Drive: they survive runtime resets and are shared with the FP8 notebook
# (every notebook gets its own Colab runtime, and /content is not shared between runtimes).
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/kv-cache/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Config
Only the multikey task at 16K, 5 variants per depth, FP16 (reused) and INT4.

In [ ]:
MODEL = "Qwen/Qwen3-4B"
TASKS = ['multikey']         # run in this order; each task gets its own result files
N_DISTRACTORS = 3                      # multikey only: needles besides the queried one (RULER niah_multikey_1: 4 in total)
TASK = TASKS[0]                        # current task; the loops below set it

SHORT = MODEL.split("/")[-1]
CFG = dict(
    model_id=MODEL,
    lengths=[16384],
    depths=[0, 25, 50, 75, 100],     # % of the haystack before the queried needle
    n_variants=5,                   # needles (key, value, haystack start) per (length, depth) cell
    max_new_tokens=128,              # RULER generates up to 128 tokens for NIAH
    seed=0,
    tag="fp16",
)
MODES = ["fp16", "int4"]                # fp16 rows already exist from the v2 run and are skipped
PREFILL_EXACT = True
SMOKE = dict(lengths=[2048], depths=[0, 50, 100], n_variants=2)
def rpath(mode): return f"{RESULTS_DIR}/ruler_{TASK}_{SHORT}_{mode}.jsonl"
def smoke_path(): return f"{RESULTS_DIR}/ruler_{TASK}_{SHORT}_smoke.jsonl"
print(SHORT, TASKS, CFG["lengths"], "variants", CFG["n_variants"])

## Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CFG["model_id"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], torch_dtype=torch.float16, device_map="cuda", attn_implementation="sdpa"
).eval()

mc = model.config
head_dim = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
KV_BYTES_PER_TOKEN = 2 * mc.num_hidden_layers * mc.num_key_value_heads * head_dim * 2   # FP16
weights_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
print(f"layers {mc.num_hidden_layers}, q heads {mc.num_attention_heads}, kv heads {mc.num_key_value_heads}, head_dim {head_dim}")
print(f"weights {weights_gib:.2f} GiB, KV cache {KV_BYTES_PER_TOKEN/1024:.0f} KiB/token "
      f"-> {KV_BYTES_PER_TOKEN*16384/2**30:.2f} GiB at 16K")

# sanity check
msgs = [{"role": "user", "content": "Say hello in five words."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                              return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**ids, max_new_tokens=16, do_sample=False, temperature=None, top_p=None, top_k=None)
print(repr(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)))

## Haystack: Paul Graham essays
The same essay folder RULER and the original NIAH test use (from the LLMTest_NeedleInAHaystack repository). Essays are concatenated and split into sentences; each variant starts at a different sentence and wraps around if needed.

In [ ]:
import glob, subprocess
ESSAY_DIR = "/content/niah/needlehaystack/PaulGrahamEssays"
if not os.path.isdir(ESSAY_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/gkamradt/LLMTest_NeedleInAHaystack", "/content/niah"], check=True)
files = sorted(glob.glob(f"{ESSAY_DIR}/*.txt"))
raw = " ".join(open(f, encoding="utf-8").read() for f in files)
raw = re.sub(r"\s+", " ", raw).strip()
SENTS = [s for s in re.split(r"(?<=[.!?])\s+", raw) if s]
SENT_TOK = [len(x) for x in tok(SENTS, add_special_tokens=False)["input_ids"]]
print(len(files), "essays,", len(SENTS), "sentences,", sum(SENT_TOK), "tokens")
assert sum(SENT_TOK) > 2 * max(CFG["lengths"]), "haystack too short"

## Needles and prompt builder
- Each variant draws its own keys and values (seeded), so a variant is the same needle set at every length and depth.
- The queried needle goes before sentence `k` at the requested depth. Distractors (multikey) go at seeded random positions that do not depend on the depth.
- Template and answer prefix are RULER's, verbatim (the answer prefix says "are", as in RULER).

In [ ]:
from wonderwords import random_word
_adjs = random_word._get_words_from_text_file("adjectivelist.txt")
_nouns = random_word._get_words_from_text_file("nounlist.txt")
WORDS = sorted(set(f"{a}-{n}" for a in _adjs for n in _nouns))

TEMPLATE = ("A special magic number is hidden within the following text. Make sure to memorize it. "
            "I will quiz you about the number afterwards.\n{context}\n"
            "What is the special magic number for {query} mentioned in the provided text?")
ANSWER_PREFIX = "The special magic number for {query} mentioned in the provided text are"
NEEDLE = "One of the special magic numbers for {key} is: {value}."

def ntok(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def build_prompt(target_len, depth, variant, task=None, seed=CFG["seed"]):
    task = task or TASK
    n_needles = 1 + (N_DISTRACTORS if task == "multikey" else 0)
    r = random.Random(f"needles-{seed}-{variant}")
    keys = r.sample(WORDS, n_needles)
    values = []
    while len(values) < n_needles:
        x = str(r.randint(1000000, 9999999))
        if x not in values: values.append(x)
    needles = [NEEDLE.format(key=k, value=v) for k, v in zip(keys, values)]
    query, value = keys[0], values[0]

    overhead = ntok(TEMPLATE.format(context="", query=query)) + ntok(ANSWER_PREFIX.format(query=query)) + 30
    budget = target_len - overhead - sum(ntok(n) for n in needles)
    rng = random.Random(f"hay-{seed}-{target_len}-{variant}")
    start = rng.randrange(len(SENTS))
    idx, total = [], 0
    while total < budget:
        i = (start + len(idx)) % len(SENTS)
        idx.append(i); total += SENT_TOK[i]
    chosen = [SENTS[i] for i in idx]

    k = round(depth / 100 * len(chosen))
    actual_depth = 100 * sum(SENT_TOK[i] for i in idx[:k]) / total
    pr = random.Random(f"pos-{seed}-{target_len}-{variant}")
    inserts = [(k, 0, needles[0])] + [(pr.randrange(len(chosen) + 1), 1, n) for n in needles[1:]]
    out, ins = [], sorted(inserts)
    j = 0
    for pos in range(len(chosen) + 1):
        while j < len(ins) and ins[j][0] == pos:
            out.append(ins[j][2]); j += 1
        if pos < len(chosen): out.append(chosen[pos])
    context = " ".join(out)
    prompt = TEMPLATE.format(context=context, query=query)
    return prompt, dict(value=value, key=query, prefix=ANSWER_PREFIX.format(query=query),
                        distractors=values[1:], actual_depth=actual_depth)

# checks: each value appears exactly once, length close to the target, all needles present
for task in ["single", "multikey"]:
    p, m = build_prompt(4096, 50, 0, task=task)
    assert p.count(m["value"]) == 1 and all(p.count(v) == 1 for v in m["distractors"])
    print(task, "| tokens", ntok(p), "| depth", round(m["actual_depth"], 1), "| key", m["key"], "| distractors", len(m["distractors"]))
print(p[:300], "...", p[-250:], "\n[assistant starts with]", m["prefix"])

## Generation
The chat template is applied to RULER's input, and RULER's answer prefix is appended as the start of the assistant turn. The cache argument is `cache` in transformers 5.x and `past_key_values` in 4.x.

In [ ]:
import transformers
CACHE_KW = "cache" if int(transformers.__version__.split(".")[0]) >= 5 else "past_key_values"

@torch.no_grad()
def answer(prompt, prefix, max_new_tokens, cache=None):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False) + prefix
    ids = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    kw = {CACHE_KW: cache} if cache is not None else {}
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None, **kw)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated()
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), ids.shape[1], dt, peak

## Trial and grid runner
One JSON line per trial, resumes after a disconnect, marks OOM lengths. `distractor_hit` = the output contains another needle's value (a retrieval error, multikey only).

In [ ]:
def run_trial(L, d, v, make_cache=None):
    prompt, meta = build_prompt(L, d, v)
    cache = make_cache() if make_cache is not None else None
    out_text, n_in, dt, peak = answer(prompt, meta["prefix"], CFG["max_new_tokens"], cache)
    correct = meta["value"].lower() in out_text.lower()          # RULER string_match_all, one reference
    return dict(tag=CFG["tag"], task=TASK, length=L, depth=d, variant=v, n_tokens=n_in,
                actual_depth=round(meta["actual_depth"], 1), key=meta["key"], expected=meta["value"],
                output=out_text.strip(), correct=bool(correct),
                distractor_hit=any(x in out_text for x in meta["distractors"]),
                seconds=round(dt, 2), peak_gib=round(peak / 2**30, 2), status="ok")

def run_grid(lengths, depths, n_variants, path, make_cache=None, retry_oom=False):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            if r["status"] == "ok" or not retry_oom:
                done.add((r["length"], r["depth"], r["variant"]))
    todo = [(L, d, v) for L in lengths for d in depths for v in range(n_variants) if (L, d, v) not in done]
    oom_at = []
    with open(path, "a") as f:
        for L, d, v in tqdm(todo):
            if any(L >= x for x in oom_at):
                rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            else:
                try:
                    rec = run_trial(L, d, v, make_cache)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); oom_at.append(L)
                    print(f"OOM at length {L}; marking this and longer lengths as oom")
                    rec = dict(tag=CFG["tag"], length=L, depth=d, variant=v, status="oom")
            f.write(json.dumps(rec) + "\n"); f.flush()
    return pd.read_json(path, lines=True)

## Quantizers
FP8 e4m3 / e5m2 round-trip (clamped to the format's range), with a check that the native cast matches an exact emulation. INT8/INT4 are kept only as optional references.

In [ ]:
FP8_FMT = {"fp8_e4m3": dict(dtype="float8_e4m3fn", mb=3, emin=-6, vmax=448.0),
           "fp8_e5m2": dict(dtype="float8_e5m2",   mb=2, emin=-14, vmax=57344.0)}

def fp8_native(x, fmt):
    c = FP8_FMT[fmt]
    return x.float().clamp(-c["vmax"], c["vmax"]).to(getattr(torch, c["dtype"])).to(x.dtype)

def fp8_emulated(x, fmt):
    c = FP8_FMT[fmt]
    xf = x.float().clamp(-c["vmax"], c["vmax"])
    e = torch.floor(torch.log2(xf.abs().clamp(min=2.0 ** (c["emin"] - c["mb"]))))
    e = e.clamp(min=c["emin"])
    step = 2.0 ** (e - c["mb"])
    return (torch.round(xf / step) * step).clamp(-c["vmax"], c["vmax"]).to(x.dtype)

def quant_int(x, bits):
    xf = x.float()
    mn, mx = xf.amin(-1, keepdim=True), xf.amax(-1, keepdim=True)
    qmax = 2 ** bits - 1
    scale = ((mx - mn) / qmax).clamp(min=1e-8)
    return (torch.round((xf - mn) / scale).clamp(0, qmax) * scale + mn).to(x.dtype)

# choose native or emulated FP8 for this GPU
probe = torch.cat([torch.randn(1 << 15), torch.randn(1 << 15) * 1e-3, torch.randn(1 << 15) * 100]).half().cuda()
IMPL = {}
for fmt in FP8_FMT:
    emu = fp8_emulated(probe, fmt)
    try:
        nat = fp8_native(probe, fmt)
        IMPL[fmt] = fp8_native
        print(f"{fmt}: native cast works; differs from emulation in {int((nat != emu).sum())} of {probe.numel()} values")
    except Exception as e:
        IMPL[fmt] = fp8_emulated
        print(f"{fmt}: native cast failed ({type(e).__name__}); using emulation")

def get_qfn(mode):
    if mode in FP8_FMT:
        f = IMPL[mode]; return lambda x: f(x, mode)
    if mode == "int8": return lambda x: quant_int(x, 8)
    if mode == "int4": return lambda x: quant_int(x, 4)
    raise ValueError(mode)

# relative error on random data, as a rough scale of how lossy each mode is
x = torch.randn(1, 8, 512, 128, device="cuda").half()
for mode in ["fp8_e4m3", "fp8_e5m2", "int8", "int4"]:
    q = get_qfn(mode)(x)
    print(f"{mode:9s} relative error {((q.float() - x.float()).norm() / x.float().norm()).item():.4f}")

## Quantizing cache
Every K/V written to the cache goes through the quantizer. With `PREFILL_EXACT`, the prompt's own attention uses exact K/V and the stored cache is quantized, so every decode step reads quantized values.

In [ ]:
from transformers import DynamicCache

class QuantCache(DynamicCache):
    def __init__(self, qfn, prefill_exact=True, record=False):
        try:
            super().__init__(config=model.config)
        except TypeError:
            super().__init__()
        self.qfn, self.prefill_exact = qfn, prefill_exact
        self.stats = [] if record else None

    def update(self, key_states, value_states, layer_idx, *args, **kwargs):
        first = self.get_seq_length(layer_idx) == 0
        qk, qv = self.qfn(key_states), self.qfn(value_states)
        if self.stats is not None:
            kf, vf = key_states.float(), value_states.float()
            self.stats.append((((qk.float() - kf).norm() / kf.norm()).item(),
                               ((qv.float() - vf).norm() / vf.norm()).item()))
        k, v = super().update(qk, qv, layer_idx, *args, **kwargs)
        if first and self.prefill_exact:
            return key_states, value_states
        return k, v

def qcache(mode, record=False):
    return lambda: QuantCache(get_qfn(mode), PREFILL_EXACT, record)

# sanity: one real prompt per mode, K/V error actually seen
prompt, meta = build_prompt(1024, 50, 0)
for mode in ["fp8_e4m3", "fp8_e5m2"]:
    c = qcache(mode, record=True)()
    out, n_in, dt, peak = answer(prompt, meta["prefix"], 16, c)
    s = np.array(c.stats)
    print(f"{mode:9s} K rel. error {s[:, 0].mean():.4f}, V rel. error {s[:, 1].mean():.4f} | answer: {out.strip()!r} (expected {meta['value']})")

## Smoke test
Six FP16 trials per task: check the answers and the time per trial before the full run.

In [ ]:
CFG["tag"] = "fp16"
for TASK in TASKS:
    sm = run_grid(SMOKE["lengths"], SMOKE["depths"], SMOKE["n_variants"], smoke_path())
    print(f"--- {TASK}")
    print(sm[["length", "n_tokens", "depth", "variant", "expected", "output", "correct", "seconds", "peak_gib"]].to_string())

## Run all tasks and modes
Task by task (in the order of `TASKS`), FP16 first, then each FP8 mode on the full grid. Each trial is appended to its file on Drive and the Drive is flushed after each mode; rerunning resumes.

In [ ]:
def run_mode(mode):
    CFG["tag"] = mode
    mk = None if mode == "fp16" else qcache(mode)
    return run_grid(CFG["lengths"], CFG["depths"], CFG["n_variants"], rpath(mode), make_cache=mk)

def save_to_drive():
    # Colab syncs Drive writes lazily; unmount forces the sync, then remount so the next mode can write
    drive.flush_and_unmount()
    drive.mount("/content/drive", force_remount=True)

for TASK in TASKS:
    for mode in MODES:
        d = run_mode(mode)
        ok = d[d.status == "ok"]
        per_len = ok.groupby("length").correct.mean().round(2).to_dict()
        print(f"{TASK} {mode}: {int(ok.correct.sum())}/{len(ok)} correct, {int((d.status == 'oom').sum())} oom | by length {per_len}")
        save_to_drive()
        print(f"saved {rpath(mode)}")

## Results and paired comparison

In [ ]:
def summarize(df):
    ok = df[df.status == "ok"]
    return ok.groupby(["length", "depth"])["correct"].agg(k="sum", n="size").reset_index()

def plot_heatmap(df, title, fname):
    g = summarize(df)
    lengths, depths = sorted(g.length.unique()), sorted(g.depth.unique())
    M = np.full((len(depths), len(lengths)), np.nan)
    for r in g.itertuples():
        M[depths.index(r.depth), lengths.index(r.length)] = r.k / r.n
    fig, ax = plt.subplots(figsize=(1.5 * len(lengths) + 2, 0.7 * len(depths) + 1.8))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(lengths))); ax.set_xticklabels([f"{L//1024}K" for L in lengths])
    ax.set_yticks(range(len(depths)));  ax.set_yticklabels([f"{d}%" for d in depths])
    ax.set_xlabel("context length (tokens)"); ax.set_ylabel("needle depth")
    for r in g.itertuples():
        ax.text(lengths.index(r.length), depths.index(r.depth), f"{int(r.k)}/{int(r.n)}",
                ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="success rate")
    ax.set_title(title, fontsize=10)
    fig.savefig(fname, dpi=160, bbox_inches="tight"); plt.show()

In [ ]:
for TASK in TASKS:
    print(f"\n========== {TASK} ==========")
    results = {m: pd.read_json(rpath(m), lines=True) for m in MODES if os.path.exists(rpath(m))}
    assert "fp16" in results, "FP16 results missing"
    f16 = results["fp16"]
    f16 = f16[f16.status == "ok"][["length", "depth", "variant", "correct"]].rename(columns={"correct": "fp16"})

    rows = []
    for mode, d in results.items():
        plot_heatmap(d, f"RULER {TASK}, {mode} cache, {SHORT}, {GPU_NAME}", f"{RESULTS_DIR}/ruler_{TASK}_{SHORT}_{mode}_heatmap.png")
        if mode == "fp16":
            continue
        q = d[d.status == "ok"].rename(columns={"correct": "quant"})
        m = q.merge(f16, on=["length", "depth", "variant"], how="inner")
        for L, g in m.groupby("length"):
            rows.append(dict(mode=mode, length=L, n=len(g), fp16=int(g.fp16.sum()), quant=int(g.quant.sum()),
                             fp16_only=int((g.fp16 & ~g.quant).sum()), quant_only=int((~g.fp16 & g.quant).sum())))
    print()
    print(pd.DataFrame(rows).to_string(index=False))

## Paired significance and error types
Exact McNemar test on the discordant pairs, per length and in total. Error types: *retrieval: distractor value* = answered another needle's value (multikey); *copy* = starts with the right digits but dropped, truncated or changed some; *retrieval* = no number or an unrelated one.

In [ ]:
import re, math
from collections import Counter

def mcnemar_p(b, c):
    n = b + c
    if n == 0: return 1.0
    return min(1.0, 2 * sum(math.comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)

def err_type(expected, output, distractors=()):
    if any(x in output for x in distractors):
        return "retrieval: distractor value"
    digits = re.sub(r"\D", "", output)
    if not digits:
        return "retrieval: no number"
    if expected.startswith(digits):
        return "copy: truncated prefix"
    it = iter(expected)
    if len(digits) < len(expected) and all(ch in it for ch in digits):
        return "copy: dropped digits"
    if digits[:2] == expected[:2]:
        return "copy: other (right start)"
    return "retrieval: unrelated number"

for TASK in TASKS:
    print(f"\n========== {TASK} ==========")
    R = {m: pd.read_json(rpath(m), lines=True, dtype=False) for m in MODES if os.path.exists(rpath(m))}
    base = R["fp16"].query("status == 'ok'")[["length", "depth", "variant", "correct"]].rename(columns={"correct": "fp16"})
    rows = []
    for m in [x for x in R if x != "fp16"]:
        q = R[m].query("status == 'ok'")[["length", "depth", "variant", "correct"]].rename(columns={"correct": "q"})
        j = q.merge(base, on=["length", "depth", "variant"])
        for L, g in list(j.groupby("length")) + [("all", j)]:
            b, c = int((g.fp16 & ~g.q).sum()), int((~g.fp16 & g.q).sum())
            rows.append(dict(mode=m, length=L, n=len(g), fp16=int(g.fp16.sum()), quant=int(g.q.sum()),
                             fp16_only=b, quant_only=c, p=round(mcnemar_p(b, c), 4)))
    print(pd.DataFrame(rows).to_string(index=False))

    W = []
    for m, d in R.items():
        w = d.query("status == 'ok' and not correct").copy()
        w["mode"] = m
        w["type"] = [err_type(str(e), str(o), build_prompt(L, dd, v)[1]["distractors"]) for e, o, L, dd, v in zip(w.expected, w.output, w.length, w.depth, w.variant)]
        W.append(w)
    W = pd.concat(W, ignore_index=True)
    print("\nerror types per mode")
    print(pd.crosstab(W["mode"], W["type"]).to_string())
    key = W.groupby(["length", "depth", "variant"])["mode"].apply(lambda s: "+".join(sorted(s)))
    print("\nwhich modes fail on the same prompt:", Counter(key).most_common())

## Save to Drive

In [ ]:
save_to_drive()
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith("ruler_") and SHORT in f:
        print(f"{f:60s} {os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1024:8.1f} KiB")

## Save this notebook (with outputs) to Drive
Uses an internal Colab call; if it fails, use File > Save a copy in Drive.

In [ ]:
import json, time
from google.colab import _message

nb_json = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
nb_path = f"{RESULTS_DIR}/int4_check_{SHORT}_run_{time.strftime('%Y%m%d_%H%M')}.ipynb"
with open(nb_path, "w") as f:
    json.dump(nb_json, f)
save_to_drive()
print(f"saved {nb_path}: {len(nb_json['cells'])} cells, {os.path.getsize(nb_path) / 1024:.0f} KiB")

## For the report
- One sentence: on the same 25 multikey prompts at 16K, Qwen3-4B with an INT4 cache scores X/25 against FP16 25/25, so the test does detect a cache loss on this model; FP8 e4m3 and e5m2 were 25/25 on these prompts (and 200/200 overall).
- INT4 here is per-token min-max over head_dim for both K and V, simulated; real INT4 KV caches (KIVI) quantize K per channel, so this is a deliberately crude variant.